<h1><center>Tokenizing With Code</center></h1>

In [1]:
# tiktoken lets us see exactly how OpenAI's models break text into tokens
import tiktoken
import os 
from dotenv import load_dotenv
from openai import OpenAI

In [2]:

# Get the tokenizer that matches gpt-4.1-mini's vocabulary, then encode a
# sentence into its list of token IDs (this doesn't call the API - it's local)
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
tokens = encoding.encode("Hi my name is Michael and I like playing golf")


In [3]:
# Raw token IDs - each integer maps to a chunk of text in the model's vocabulary
tokens

[12194, 922, 1308, 382, 13266, 326, 357, 1299, 8252, 18340]

In [4]:
# Decode each token ID individually to see what text piece it corresponds to
# (note the leading spaces are part of the token itself, not added by print)
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
13266 =  Michael
326 =  and
357 =  I
1299 =  like
8252 =  playing
18340 =  golf


In [5]:
# Decoding a single token ID by itself - useful for spot-checking the vocabulary
encoding.decode([326])

' and'

## And another topic!

In [6]:
# Load environment variables from .env and sanity-check the OpenAI API key
# before we try to use it (fails fast with a clear message instead of a
# confusing API error later)
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found - please head over to the troubleshooting notebook in this folder to identify and fix...")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right API key- see troubleshooting notebook.")
else: 
    print("API key is valid.")

API key is valid.


In [7]:
# Client picks up OPENAI_API_KEY from the environment automatically
openai = OpenAI()

## A message to OpenAI is a list of dictionaries

In [8]:
# The Chat Completions API expects a list of messages, each with a "role"
# (system/user/assistant) and "content". "system" sets the assistant's
# behavior/persona; "user" is the human's message.
messages = [
    {
    "role":"system", 
    "content": "You are a helpful assistant."
    }, 
    {
        "role":"user",
        "content": "Hello, My name is Michael."
    }
] 

In [9]:
# Send the message list to the model and print the assistant's reply text
response = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)

response.choices[0].message.content

'Hello Michael! How can I assist you today?'

## Ok let's now ask a follow up question

In [10]:
# This is a brand-new messages list with no history of the earlier
# conversation - the API is stateless, so the model has no memory of
# "Michael" unless we include it again ourselves.
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "What's my name?"
    }
]

In [11]:
# As expected, the model can't answer - it has no memory of the prior turn
response = openai.chat.completions.create(
    model = "gpt-4.1-mini", messages = messages
)

response.choices[0].message.content

'I’m sorry, but I don’t know your name. Could you please tell me?'

In [12]:
# To give the model "memory," we manually include the full conversation
# history in the messages list, including the assistant's own prior reply.
# This is how chat context is maintained across turns.
messages = [
    {
        "role": "system", 
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Hi I'm Michael." 

    },
    {
        "role": "assistant",
        "content": "Hi, Michael. How can I assist you today?" 
    },
    {
        "role": "user",
        "content": "What's my name?"
    }
]

In [13]:
# Now the model correctly answers "Michael" since the name is present
# earlier in the messages list we sent
response = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)

response.choices[0].message.content

'Your name is Michael. How can I help you today?'